# Patchscope Activation Interpretation Analysis

Does injecting a persona's hidden-state activation into an interpretation pass reveal **privileged information** about that persona's internal reasoning?

Key metrics:
- **source_gain**: logit shift toward source answer (real vs baseline)
- **shuffled_gain**: logit shift from shuffled activation (noise floor)
- **net_gain**: source_gain − shuffled_gain (question-specific signal)
- Cross-evaluator comparison: same-persona vs neutral vs opposite-persona

## 1. Setup

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
sns.set_style('whitegrid')

RESULTS_PATH = '../results/raw/patchscope_llama-3-1-8b_20260322_224126_matrix_cat1and4_.jsonl'

def _extract_model_name(path: str) -> str:
    """Extract model name from patchscope results filename."""
    name = Path(path).stem.lower()
    # Remove prefix/suffix
    for prefix in ['patchscope_', 'patchscope_dual_test_', 'patchscope_test_']:
        if name.startswith(prefix):
            name = name[len(prefix):]
            break
    # Remove timestamp and suffix (e.g. _20260322_224126_matrix_cat1and4_)
    import re
    name = re.sub(r'_\d{8}_\d{6}.*', '', name)
    # Map known model slugs
    model_map = {
        'llama-3-1-8b': 'Llama 3.1 8B Instruct',
        'llama-3-3-70b': 'Llama 3.3 70B Instruct',
        'tinyllama-1-1b--v1-0': 'TinyLlama 1.1B',
        'dolphin-2-9-4-llama3-1-8b': 'Dolphin 2.9.4 Llama 8B',
    }
    return model_map.get(name, name.replace('-', ' ').title())

MODEL = _extract_model_name(RESULTS_PATH)
print(f'Model: {MODEL}')

# Resolve path
_p = Path(RESULTS_PATH)
if not _p.exists():
    for base in [Path.cwd(), Path.cwd().parent]:
        candidate = base / RESULTS_PATH
        if candidate.exists():
            RESULTS_PATH = str(candidate)
            break
assert Path(RESULTS_PATH).exists(), f'Not found: {RESULTS_PATH}'

## 2. Load & Prepare Data

In [ ]:
records = []
with open(RESULTS_PATH) as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'Loaded {len(df)} records')
print(f'Questions: {df["question_id"].nunique()}, Layer pairs: {df.groupby(["source_layer","injection_layer"]).ngroups}')
print(f'Conditions: {df["condition"].value_counts().to_dict()}')

# Extract logit for source answer
def get_source_answer_logit(row):
    sa = row.get('source_direct_answer')
    logits = row.get('choice_logits')
    if sa and logits and sa in logits:
        return logits[sa]
    return np.nan

df['source_answer_logit'] = df.apply(get_source_answer_logit, axis=1)
df['layer_pair'] = df['source_layer'].astype(str) + '\u2192' + df['injection_layer'].astype(str)

def eval_relationship(row):
    if row['evaluator_persona'] == 'neutral_evaluator':
        return 'neutral'
    elif row['evaluator_persona'] == row['source_persona']:
        return 'same'
    else:
        return 'opposite'

df['eval_relation'] = df.apply(eval_relationship, axis=1)
print(f'Evaluator relations: {df["eval_relation"].value_counts().to_dict()}')

## 3. Compute Source Gain Metrics

For each (question, source_persona, evaluator, layer_pair), compute:
- **source_gain** = logit_real(source_answer) - logit_baseline(source_answer)
- **shuffled_gain** = logit_shuffled(source_answer) - logit_baseline(source_answer)
- **net_gain** = source_gain - shuffled_gain (question-specific signal above persona noise)

In [ ]:
key_cols = ['question_id', 'source_persona', 'evaluator_persona', 'source_layer', 'injection_layer']

def build_gain_df(df):
    ae = df[df['template_name'] == 'answer_extraction'].copy()
    groups = {}
    for _, row in ae.iterrows():
        key = tuple(row[c] for c in key_cols)
        groups.setdefault(key, {})[row['condition']] = row

    gains = []
    for key, conds in groups.items():
        if 'real' not in conds or 'text_only_baseline' not in conds:
            continue
        real = conds['real']
        baseline = conds['text_only_baseline']
        shuffled = conds.get('shuffled')
        src_answer = real['source_direct_answer']
        if not src_answer:
            continue
        real_logits = real.get('choice_logits', {})
        base_logits = baseline.get('choice_logits', {})
        if not real_logits or not base_logits or src_answer not in real_logits:
            continue

        source_gain = real_logits[src_answer] - base_logits[src_answer]
        base_top = max(base_logits, key=base_logits.get)
        target_drop = real_logits[base_top] - base_logits[base_top]

        shuf_gain = np.nan
        if shuffled and shuffled.get('choice_logits'):
            sl = shuffled['choice_logits']
            if src_answer in sl:
                shuf_gain = sl[src_answer] - base_logits[src_answer]

        eval_p = key[2]
        if eval_p == 'neutral_evaluator':
            rel = 'neutral'
        elif eval_p == key[1]:
            rel = 'same'
        else:
            rel = 'opposite'

        gains.append({
            'question_id': key[0], 'source_persona': key[1],
            'evaluator_persona': key[2],
            'source_layer': key[3], 'injection_layer': key[4],
            'layer_pair': f'{key[3]}\u2192{key[4]}',
            'source_direct_answer': src_answer,
            'real_predicted': real.get('predicted'),
            'baseline_predicted': baseline.get('predicted'),
            'source_gain': source_gain,
            'target_drop': target_drop,
            'shuffled_gain': shuf_gain,
            'net_gain': source_gain - shuf_gain if not np.isnan(shuf_gain) else np.nan,
            'real_correct': real.get('predicted') == src_answer,
            'baseline_correct': baseline.get('predicted') == src_answer,
            'eval_relation': rel,
        })
    return pd.DataFrame(gains)

gdf = build_gain_df(df)
print(f'Gain records: {len(gdf)}')
print(f'\nSource gain summary:')
display(gdf[['source_gain', 'shuffled_gain', 'net_gain']].describe().round(3))

## 4. Source Gain by Layer Pair

Which layer pairs best transfer source answer information?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
lp_order = gdf.groupby('layer_pair')['source_gain'].mean().sort_values(ascending=False).index.tolist()

sns.barplot(data=gdf, x='layer_pair', y='source_gain', order=lp_order, ax=axes[0], ci=95, color='steelblue')
axes[0].set_title('Source Gain (real \u2212 baseline)')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=gdf, x='layer_pair', y='shuffled_gain', order=lp_order, ax=axes[1], ci=95, color='salmon')
axes[1].set_title('Shuffled Gain (noise floor)')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(data=gdf, x='layer_pair', y='net_gain', order=lp_order, ax=axes[2], ci=95, color='seagreen')
axes[2].set_title('Net Gain (real \u2212 shuffled)')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].tick_params(axis='x', rotation=45)

fig.suptitle(f'Source Answer Logit Shifts by Layer Pair \u2014 {MODEL}', fontsize=13)
plt.tight_layout()
plt.show()

lp_stats = gdf.groupby('layer_pair').agg(
    source_gain_mean=('source_gain', 'mean'),
    source_gain_std=('source_gain', 'std'),
    shuffled_gain_mean=('shuffled_gain', 'mean'),
    net_gain_mean=('net_gain', 'mean'),
    pct_positive_gain=('source_gain', lambda x: (x > 0).mean()),
    real_accuracy=('real_correct', 'mean'),
    baseline_accuracy=('baseline_correct', 'mean'),
    n=('source_gain', 'count'),
).round(3).sort_values('net_gain_mean', ascending=False)
display(lp_stats)

## 5. Privileged Access: Cross-Evaluator Comparison

The key test: does the **same-persona** evaluator show larger source_gain than **neutral** or **opposite**?

If personas have privileged access: same > neutral > opposite

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
eval_order = ['same', 'neutral', 'opposite']

sns.barplot(data=gdf, x='eval_relation', y='source_gain', order=eval_order, ax=axes[0], ci=95, palette='Set2')
axes[0].set_title('Source Gain by Evaluator Relation')
axes[0].axhline(0, color='black', linewidth=0.5)

sns.barplot(data=gdf, x='eval_relation', y='shuffled_gain', order=eval_order, ax=axes[1], ci=95, palette='Set2')
axes[1].set_title('Shuffled Gain by Evaluator Relation')
axes[1].axhline(0, color='black', linewidth=0.5)

sns.barplot(data=gdf, x='eval_relation', y='net_gain', order=eval_order, ax=axes[2], ci=95, palette='Set2')
axes[2].set_title('Net Gain by Evaluator Relation')
axes[2].axhline(0, color='black', linewidth=0.5)

fig.suptitle(f'Privileged Access Test: Same vs Neutral vs Opposite \u2014 {MODEL}', fontsize=13)
plt.tight_layout()
plt.show()

eval_stats = gdf.groupby('eval_relation').agg(
    source_gain_mean=('source_gain', 'mean'),
    shuffled_gain_mean=('shuffled_gain', 'mean'),
    net_gain_mean=('net_gain', 'mean'),
    real_accuracy=('real_correct', 'mean'),
    baseline_accuracy=('baseline_correct', 'mean'),
    n=('source_gain', 'count'),
).round(4).loc[eval_order]
display(eval_stats)

## 6. Cross-Evaluator \u00d7 Layer Pair Heatmap

Does the evaluator relationship effect vary by layer pair?

In [ ]:
pivot = gdf.pivot_table(values='net_gain', index='layer_pair', columns='eval_relation', aggfunc='mean')[eval_order]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title(f'Net Gain (real\u2212shuffled) by Layer Pair \u00d7 Evaluator \u2014 {MODEL}')
plt.tight_layout()
plt.show()

## 7. Evaluator Persona Dominance

How often does the evaluator's baseline prior override the injected source activation?

In [ ]:
gdf['source_baseline_agree'] = gdf['source_direct_answer'] == gdf['baseline_predicted']
disagree = gdf[~gdf['source_baseline_agree']].copy()
disagree['flipped_to_source'] = disagree['real_predicted'] == disagree['source_direct_answer']

print(f'Source and baseline agree: {gdf["source_baseline_agree"].mean():.1%}')
print(f'Source and baseline disagree: {(~gdf["source_baseline_agree"]).mean():.1%}')
print(f'\nWhen they disagree, injection flips to source answer: {disagree["flipped_to_source"].mean():.1%}')

eval_order = ['same', 'neutral', 'opposite']
print('\nFlip rate by evaluator relation (disagreement trials):')
for rel in eval_order:
    sub = disagree[disagree['eval_relation'] == rel]
    if len(sub) > 0:
        print(f'  {rel:>10s}: {sub["flipped_to_source"].mean():.1%} ({len(sub)} trials)')

print(f'\nMean source_gain on agree vs disagree trials:')
print(f'  Agree:    {gdf[gdf["source_baseline_agree"]]["source_gain"].mean():.3f}')
print(f'  Disagree: {disagree["source_gain"].mean():.3f}')

## 8. Source Gain Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, rel in zip(axes, eval_order):
    sub = gdf[gdf['eval_relation'] == rel]
    ax.hist(sub['source_gain'].dropna(), bins=40, alpha=0.6, color='steelblue', label='source_gain')
    ax.hist(sub['shuffled_gain'].dropna(), bins=40, alpha=0.6, color='salmon', label='shuffled_gain')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.axvline(sub['source_gain'].mean(), color='steelblue', linestyle='--', linewidth=1.5)
    ax.axvline(sub['shuffled_gain'].mean(), color='salmon', linestyle='--', linewidth=1.5)
    ax.set_title(f'{rel} evaluator')
    ax.set_xlabel('Logit shift')
    ax.legend(fontsize=8)

fig.suptitle(f'Source Gain vs Shuffled Gain Distributions \u2014 {MODEL}', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Same-Layer vs Cross-Layer Pairs

The toy Bezos test suggested late\u2192early pairs are best for semantic decoding.
For MCQ answer extraction with evaluator prompts, does this still hold?

In [ ]:
gdf['pair_type'] = np.where(gdf['source_layer'] == gdf['injection_layer'], 'same-layer', 'cross-layer')

pair_comp = gdf.groupby(['pair_type', 'eval_relation']).agg(
    source_gain_mean=('source_gain', 'mean'),
    net_gain_mean=('net_gain', 'mean'),
    real_accuracy=('real_correct', 'mean'),
    n=('source_gain', 'count'),
).round(4)
display(pair_comp)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric, title in zip(axes, ['source_gain', 'net_gain'], ['Source Gain', 'Net Gain']):
    sns.barplot(data=gdf, x='pair_type', y=metric, hue='eval_relation',
                hue_order=eval_order, ax=ax, ci=95, palette='Set2')
    ax.set_title(title)
    ax.axhline(0, color='black', linewidth=0.5)

fig.suptitle(f'Same-Layer vs Cross-Layer Pairs \u2014 {MODEL}', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Per-Question Scatter: Source Gain vs Shuffled Gain

Points above the diagonal = question-specific signal beyond persona noise.

In [ ]:
q_avg = gdf.groupby(['question_id', 'source_persona', 'eval_relation']).agg(
    source_gain=('source_gain', 'mean'),
    shuffled_gain=('shuffled_gain', 'mean'),
    net_gain=('net_gain', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, rel in zip(axes, eval_order):
    sub = q_avg[q_avg['eval_relation'] == rel]
    ax.scatter(sub['shuffled_gain'], sub['source_gain'], alpha=0.5, s=20)
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', linewidth=0.5, alpha=0.5)
    ax.set_xlabel('Shuffled gain')
    ax.set_ylabel('Source gain')
    ax.set_title(f'{rel} evaluator (n={len(sub)})')
    above = (sub['source_gain'] > sub['shuffled_gain']).mean()
    ax.text(0.05, 0.95, f'{above:.0%} above diagonal', transform=ax.transAxes, va='top', fontsize=9)

fig.suptitle(f'Source Gain vs Shuffled Gain per Question \u2014 {MODEL}', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Full Logit Shift by Option Type

Average logit shift for source-answer options vs non-source options.

In [ ]:
ae = df[df['template_name'] == 'answer_extraction'].copy()
groups = {}
for _, row in ae.iterrows():
    key = tuple(row[c] for c in key_cols)
    groups.setdefault(key, {})[row['condition']] = row

option_shifts = []
for key, conds in groups.items():
    if 'real' not in conds or 'text_only_baseline' not in conds:
        continue
    real_l = conds['real'].get('choice_logits', {})
    base_l = conds['text_only_baseline'].get('choice_logits', {})
    src_a = conds['real']['source_direct_answer']
    if not real_l or not base_l or not src_a:
        continue
    ep = key[2]
    rel = 'neutral' if ep == 'neutral_evaluator' else ('same' if ep == key[1] else 'opposite')
    for opt in ['A', 'B', 'C', 'D']:
        if opt in real_l and opt in base_l:
            option_shifts.append({'option': opt, 'is_source_answer': opt == src_a,
                                  'eval_relation': rel, 'logit_shift': real_l[opt] - base_l[opt]})

osdf = pd.DataFrame(option_shifts)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=osdf, x='eval_relation', y='logit_shift', hue='is_source_answer',
            order=eval_order, ax=ax, ci=95, palette={True: 'steelblue', False: 'lightgray'})
ax.set_title(f'Logit Shift by Option Type \u2014 {MODEL}')
ax.set_ylabel('Mean logit shift (real \u2212 baseline)')
ax.axhline(0, color='black', linewidth=0.5)
ax.legend(title='Is source answer?', loc='upper right')
plt.tight_layout()
plt.show()

## 12. Key Findings Summary

In [ ]:
print('=' * 60)
print(f'PATCHSCOPE RESULTS SUMMARY \u2014 {MODEL}')
print('=' * 60)
print(f'\nDataset: {gdf["question_id"].nunique()} questions, {len(gdf)} gain measurements')
print(f'Layer pairs: {sorted(gdf["layer_pair"].unique())}')

print(f'\n--- Overall ---')
print(f'Mean source_gain:   {gdf["source_gain"].mean():+.3f}')
print(f'Mean shuffled_gain: {gdf["shuffled_gain"].mean():+.3f}')
print(f'Mean net_gain:      {gdf["net_gain"].mean():+.3f}')
print(f'% positive source_gain: {(gdf["source_gain"] > 0).mean():.1%}')
print(f'% positive net_gain:    {(gdf["net_gain"] > 0).mean():.1%}')

print(f'\n--- Privileged Access Test ---')
for rel in eval_order:
    sub = gdf[gdf['eval_relation'] == rel]
    print(f'  {rel:>10s}: source_gain={sub["source_gain"].mean():+.3f}  '
          f'net_gain={sub["net_gain"].mean():+.3f}  '
          f'accuracy(real)={sub["real_correct"].mean():.1%}  '
          f'accuracy(baseline)={sub["baseline_correct"].mean():.1%}')

print(f'\n--- Best Layer Pairs (by net_gain) ---')
for _, row in lp_stats.head(3).iterrows():
    print(f'  {row.name}: net_gain={row["net_gain_mean"]:+.3f}  source_gain={row["source_gain_mean"]:+.3f}')

print(f'\n--- Interpretation ---')
net_mean = gdf['net_gain'].mean()
if net_mean > 0.5:
    print('Strong evidence of question-specific information transfer beyond persona noise.')
elif net_mean > 0.1:
    print('Moderate evidence of question-specific information transfer.')
elif net_mean > 0:
    print('Weak evidence of question-specific signal. Most effect may be persona-level.')
else:
    print('No evidence of question-specific transfer. Effect is at or below shuffled noise.')

same_ng = gdf[gdf['eval_relation'] == 'same']['net_gain'].mean()
opp_ng = gdf[gdf['eval_relation'] == 'opposite']['net_gain'].mean()
if same_ng > opp_ng + 0.1:
    print('Same-persona evaluators show larger net_gain than opposite \u2192 suggestive of privileged access.')
elif abs(same_ng - opp_ng) < 0.1:
    print('Same and opposite evaluators show similar net_gain \u2192 no privileged access detected.')
else:
    print('Opposite-persona evaluators show larger net_gain \u2192 unexpected, no privileged access.')